# Day 04 — контролируемые эксперименты

## D4-01 — гипотезы, протокол и проверка reference

**Цель.** Загрузить настоящие Day02 inputs и Day03 reference, проверить их происхождение и увидеть baseline, с которым будут сравниваться эксперименты. Выход — проверенный контекст `(N, columns)` и сохранённые validation metrics. В этой ячейке нет обучения.

**Краткая теория.** Проверим три независимые гипотезы: word unigrams; замена LogisticRegression на LinearSVC с тем же TF-IDF; подбор только `LogisticRegression.C ∈ {0.1, 1, 10}`. Каждый вариант сравнивается с baseline на одних трёх `StratifiedGroupKFold` folds только внутри Day03 train. После выбора по среднему fold macro F1 выбранный pipeline будет fit на полном train и один раз проверен на прежней validation. Test не участвует в поиске и остаётся для Day05.

Pipeline заново обучает TF-IDF внутри train части каждого fold, чтобы validation части fold не влияли на vocabulary/IDF. Метрики выбора используют три класса (`labels=[0, 1, 2]`); fold standard deviation описывает разброс, но не является оценкой статистической значимости.

In [1]:
from hashlib import sha256
import json
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "SPEC.md").is_file() and (candidate / "src/finnews_sentiment").is_dir():
            return candidate
    raise FileNotFoundError("Project root with SPEC.md and src/finnews_sentiment not found")


ROOT: Path = find_project_root(Path.cwd().resolve())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from finnews_sentiment.data.baseline_config import load_baseline_config
from finnews_sentiment.data.load_data import CLASS_ORDER
from finnews_sentiment.data.load_processed import load_processed_news

print(f"Project root: {ROOT}")
print(f"Class order: {CLASS_ORDER}")

Project root: /home/makarlistkov/projects/Text-classification
Class order: ('negative', 'neutral', 'positive')


### Проверка данных и Day03 artifacts

Загрузчик проверяет схему, label mapping и Day02 provenance/checksum. Далее сверим контрольные суммы Day03, manifest и порядок validation predictions. Если артефактов нет, нужно отдельно воспроизвести Day02 и Day03 по инструкциям в `docs/day02.md` и `docs/day03.md`; notebook не скачивает данные и не подменяет их fixtures.

In [2]:
config_path = ROOT / "configs/train_baseline.json"
config = load_baseline_config(config_path)
processed_news = load_processed_news(
    ROOT / config.input_path,
    ROOT / config.provenance_path,
    ROOT / "configs/dataset.json",
)

report_dir = ROOT / "reports/day03"
manifest_path = ROOT / config.split_output_dir / "splits.csv"
artifact_paths = {
    "manifest": manifest_path,
    "split_summary": report_dir / "split_summary.json",
    "predictions": report_dir / "validation_predictions.csv",
    "metrics": report_dir / "metrics.json",
    "effective_config": report_dir / "config.json",
}
provenance = json.loads((report_dir / "provenance.json").read_text(encoding="utf-8"))
for artifact_name, artifact_path in artifact_paths.items():
    expected_hash = provenance["checksums"][artifact_name]
    actual_hash = sha256(artifact_path.read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise ValueError(f"Day03 {artifact_name} checksum mismatch")

manifest = pd.read_csv(manifest_path, dtype={"source_row_id": "int64", "group_id": "int64"})
predictions = pd.read_csv(
    artifact_paths["predictions"],
    dtype={"source_row_id": "int64", "true_label_id": "int64", "baseline_pred_label_id": "int64", "dummy_pred_label_id": "int64"},
)
split_summary = json.loads(artifact_paths["split_summary"].read_text(encoding="utf-8"))
metrics = json.loads(artifact_paths["metrics"].read_text(encoding="utf-8"))
effective_config = json.loads(artifact_paths["effective_config"].read_text(encoding="utf-8"))

if tuple(manifest.columns) != ("source_row_id", "group_id", "split"):
    raise ValueError("Day03 split manifest schema mismatch")
if tuple(predictions.columns) != ("source_row_id", "true_label_id", "baseline_pred_label_id", "dummy_pred_label_id"):
    raise ValueError("Day03 validation predictions schema mismatch")
expected_valid_ids = manifest.loc[manifest["split"] == "validation", "source_row_id"].tolist()
if predictions["source_row_id"].tolist() != expected_valid_ids:
    raise ValueError("Day03 validation predictions do not align with manifest")
if len(processed_news) != len(manifest) or manifest["source_row_id"].tolist() != processed_news["source_row_id"].tolist():
    raise ValueError("Day03 manifest does not cover processed rows in canonical order")
if metrics["evaluation_split"] != "validation" or metrics["class_order"] != list(CLASS_ORDER):
    raise ValueError("Day03 metrics split or class order mismatch")
if metrics["n_valid"] != len(predictions):
    raise ValueError("Day03 metric denominator does not match predictions")
if effective_config["random_state"] != 42 or effective_config["split_algorithm"] != "group-stratified-v1":
    raise ValueError("Day03 effective split settings differ from the agreed reference")

split_sizes = manifest["split"].value_counts().reindex(["train", "validation", "test"], fill_value=0)
print(f"Processed input: {processed_news.shape}; manifest: {manifest.shape}")
print(f"Group-aware split rows: {split_sizes.to_dict()}")
print(f"Day03 sklearn version: {provenance['versions']['scikit-learn']}")
display(pd.DataFrame([
    {"model": name, "accuracy": metrics[name]["accuracy"], "macro_f1": metrics[name]["macro_f1"], "n_valid": metrics[name]["n_samples"]}
    for name in ("baseline", "dummy")
]).set_index("model"))

Processed input: (3453, 5); manifest: (3453, 3)
Group-aware split rows: {'train': 2210, 'validation': 553, 'test': 690}
Day03 sklearn version: 1.9.1


,accuracy,macro_f1,n_valid
model,,,
baseline,0.813743,0.720965,553
dummy,0.622061,0.255667,553


### Результат и интерпретация

В сохранённой Day03 reference baseline обучался только на train, а его validation macro F1 был `0.720965` (accuracy `0.813743`). Dummy most-frequent дал macro F1 `0.255667` (accuracy `0.622061`). На validation больше neutral примеров, поэтому accuracy и macro F1 отвечают на разные вопросы; далее решения сравниваем по заранее выбранной macro F1.

Baseline — допустимый победитель Day04. CV добавляет оценку на нескольких group-aware разбиениях и выбирает настройки без использования validation; validation затем служит единственным подтверждением выбранного варианта на прежнем split. Ни CV, ни одна validation оценка не обещают улучшения на новых данных.

### Самопроверка

1. Что меняется в каждой из трёх гипотез, а что остаётся фиксированным?
2. Почему vocabulary и IDF нужно fit-ить отдельно внутри train части каждого fold?
3. Почему validation может подтвердить выбор, но не должна переизбирать конфигурацию?
4. Почему baseline может остаться лучшим, даже если проверены три направления?

## D4-02 — фиксированные Day03 splits и общие CV folds

**Цель.** Проверить и восстановить train/validation/test по сохранённому Day03 manifest, затем построить fold assignment только для outer train. Выходы: три восстановленных DataFrames, compact manifest `(N_train, 3)` с `source_row_id, group_id, fold_id` и positional train/validation indexes для каждого fold.

**Краткая теория.** Внешний split уже зафиксирован Day03. Мы не запускаем `train_test_split` повторно: иначе даже тот же seed и алгоритм могли бы дать новую membership-карту после изменения версии или входа. Внутри только `train` `StratifiedGroupKFold` делит exact-text groups на три части. Каждая train строка один раз служит CV validation; её complement — train часть соответствующего fold. `validation` и `test` Day03 остаются за пределами этой карты.

In [3]:
from finnews_sentiment.data.experiment_splits import build_train_cv_folds, load_day04_inputs

day04_inputs = load_day04_inputs(ROOT)
train_rows = day04_inputs.splits["train"]
cv_folds = build_train_cv_folds(train_rows, day04_inputs.train_manifest)

print(f"Restored rows: {len(day04_inputs.processed)}; frozen manifest: {day04_inputs.manifest.shape}")
print(f"CV train rows: {len(train_rows)}; fold manifest: {cv_folds.manifest.shape}")
print(f"StratifiedGroupKFold: n_splits={cv_folds.n_splits}; random_state={cv_folds.random_state}")
display(day04_inputs.split_summary)
display(cv_folds.summary)
display(cv_folds.manifest.head())

Restored rows: 3453; frozen manifest: (3453, 3)
CV train rows: 2210; fold manifest: (2210, 3)
StratifiedGroupKFold: n_splits=3; random_state=42


,rows,groups,negative,neutral,positive
split,,,,,
train,2210,2206,269,1374,567
validation,553,552,67,344,142
test,690,690,84,428,178


,train_rows,train_groups,validation_rows,validation_groups,train_negative_rows,train_neutral_rows,train_positive_rows,validation_negative_rows,validation_neutral_rows,validation_positive_rows
fold_id,,,,,,,,,,
0,1473,1471,737,735,179,916,378,90,458,189
1,1474,1471,736,735,180,916,378,89,458,189
2,1473,1470,737,736,179,916,378,90,458,189


,source_row_id,group_id,fold_id
0,1,1,1
1,4,4,1
2,5,5,0
3,6,6,0
4,7,7,0


### Результат и интерпретация

Day03 membership восстановлена из проверенного manifest: train — 2 210 строк/2 206 groups, validation — 553/552, test — 690/690. CV folds делят только train и имеют размеры validation 737, 736 и 737 строк (735, 735 и 736 groups). На каждой validation fold представлены все три sentiment-класса. Число строк отличается, потому что неделимы группы exact-text duplicates.

`fold_id` назначен каждой строке ровно один раз и каждой duplicate group целиком. Для CV train в fold берутся все строки с другими `fold_id`; helper проверяет непересечение групп и классы в обеих частях. Fold indices пока только структура данных: модели не fit-ились, метрик качества здесь нет.

### Самопроверка

1. Почему нельзя просто повторно вызвать Day03 `train_test_split` с тем же seed?
2. Как получить train-часть конкретного CV fold из fold assignment?
3. Почему дублированные строки одной exact-text группы должны оставаться в одной fold?
4. Какие данные гарантированно не входят в CV folds?

## D4-03 — baseline на общих CV folds

**Цель.** Оценить один baseline Pipeline на трёх заранее зафиксированных group-aware folds внутри outer train. Для каждого fold создаётся новый clone: TF-IDF vocabulary и IDF обучаются только на fold train, а метрики считаются на его validation части. Runner получает только `train_rows` и `cv_folds`; outer validation и test ему не передаются.

**Теория.** Каждый fold даёт отдельную оценку macro F1 и accuracy. Mean суммирует средний уровень по folds, std (ddof=0) показывает наблюдаемый разброс folds; это не доверительный интервал. Значение Day03 validation macro F1 `0.720965` относится к другой, единственной outer validation выборке и не является напрямую сопоставимой с CV mean. После CV выбор будет отдельно подтверждён на outer validation.

Fit/score timings и число признаков записываются как диагностика; время меняется от запуска к запуску.

In [4]:
from finnews_sentiment.models.cv_runner import run_cv_candidate
from finnews_sentiment.models.train_model import build_baseline_pipeline

baseline_cv_pipeline = build_baseline_pipeline(config)
baseline_cv = run_cv_candidate(
    candidate_id="baseline",
    pipeline=baseline_cv_pipeline,
    train=train_rows,
    fold_set=cv_folds,
)
display(baseline_cv.fold_scores)
display(pd.DataFrame([{
    "candidate_id": baseline_cv.candidate_id,
    "mean_macro_f1": baseline_cv.mean_macro_f1,
    "std_macro_f1": baseline_cv.std_macro_f1,
    "mean_accuracy": baseline_cv.mean_accuracy,
    "std_accuracy": baseline_cv.std_accuracy,
    "n_splits": baseline_cv.n_splits,
    "random_state": baseline_cv.random_state,
    "class_order": baseline_cv.class_order,
}]))

,train_rows,train_groups,validation_rows,validation_groups,n_features,macro_f1,accuracy,fit_seconds,score_seconds,convergence_status,warnings
fold_id,,,,,,,,,,,
0,1473,1471,737,735,5000,0.634554,0.785617,2.477686,0.112169,converged_no_warning,()
1,1474,1471,736,735,5000,0.698353,0.808424,2.466349,0.107574,converged_no_warning,()
2,1473,1470,737,736,5000,0.659923,0.782904,2.079011,0.095882,converged_no_warning,()


,candidate_id,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy,n_splits,random_state,class_order
0,baseline,0.664277,0.026227,0.792315,0.011444,3,42,"(negative, neutral, positive)"


### Результат и интерпретация

Таблица выше содержит по одной строке на fold, включая число train/validation строк и групп, размер обученного vocabulary, macro F1, accuracy, время fit/score и warnings. Сводка показывает mean и std macro F1/accuracy. Параметры кандидата сохранены в `baseline_cv.parameters`; исходный `baseline_cv_pipeline` остаётся unfitted, а каждый fold использовал собственный clone.

Различие между CV mean и Day03 validation metric само по себе не означает изменение качества: это разные разбиения и режимы усреднения. CV используется для выбора в Day04, outer validation — для последующей независимой confirmation выбранной конфигурации; test остаётся закрытым до D5-04.

### Самопроверка

1. Почему TF-IDF должен fit-иться внутри каждого fold, а не один раз до CV?
2. Что сообщает std fold scores и чего он не доказывает?
3. Почему CV mean нельзя напрямую сравнить с одной Day03 validation оценкой?

## D4-04 — word unigram TF-IDF

**Гипотеза.** Биграммы могут учитывать короткий контекст, но при фиксированном бюджете словаря до 5 000 признаков занимают место unigram признаков. Проверим только влияние `ngram_range=(1, 1)`; `max_features=5000`, LogisticRegression и остальные настройки baseline остаются прежними.

Оценивание проходит на тех же фиксированных train CV folds через общий runner. Сравнение попарное: для каждого fold считаем `unigram macro F1 - baseline macro F1`, затем смотрим mean/std и реальное число features. Это сравнение CV кандидатов, не Day03 validation.

In [5]:
from dataclasses import replace

from finnews_sentiment.models.train_model import build_baseline_pipeline

# Изменяется только ngram_range; прочие TF-IDF и LogReg настройки сохраняются.
unigram_tfidf = {**config.tfidf, "ngram_range": (1, 1)}
assert {key for key in config.tfidf if config.tfidf[key] != unigram_tfidf[key]} == {"ngram_range"}
unigram_config = replace(config, tfidf=unigram_tfidf)
unigram_pipeline = build_baseline_pipeline(unigram_config)
unigram_cv = run_cv_candidate(
    candidate_id="word_unigram",
    pipeline=unigram_pipeline,
    train=train_rows,
    fold_set=cv_folds,
)

paired_unigram = pd.DataFrame({
    "baseline_macro_f1": baseline_cv.fold_scores["macro_f1"],
    "unigram_macro_f1": unigram_cv.fold_scores["macro_f1"],
    "delta_macro_f1": unigram_cv.fold_scores["macro_f1"] - baseline_cv.fold_scores["macro_f1"],
    "baseline_accuracy": baseline_cv.fold_scores["accuracy"],
    "unigram_accuracy": unigram_cv.fold_scores["accuracy"],
    "baseline_features": baseline_cv.fold_scores["n_features"],
    "unigram_features": unigram_cv.fold_scores["n_features"],
})
display(paired_unigram)
display(pd.DataFrame([
    {"candidate_id": baseline_cv.candidate_id, "mean_macro_f1": baseline_cv.mean_macro_f1,
     "std_macro_f1": baseline_cv.std_macro_f1, "mean_accuracy": baseline_cv.mean_accuracy,
     "std_accuracy": baseline_cv.std_accuracy},
    {"candidate_id": unigram_cv.candidate_id, "mean_macro_f1": unigram_cv.mean_macro_f1,
     "std_macro_f1": unigram_cv.std_macro_f1, "mean_accuracy": unigram_cv.mean_accuracy,
     "std_accuracy": unigram_cv.std_accuracy},
]))
print(f"Mean paired macro F1 delta: {paired_unigram['delta_macro_f1'].mean():+.6f}")

,baseline_macro_f1,unigram_macro_f1,delta_macro_f1,baseline_accuracy,unigram_accuracy,baseline_features,unigram_features
fold_id,,,,,,,
0,0.634554,0.653685,0.019131,0.785617,0.786974,5000,4959
1,0.698353,0.697575,-0.000779,0.808424,0.804348,5000,5000
2,0.659923,0.658621,-0.001302,0.782904,0.777476,5000,5000


,candidate_id,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy
0,baseline,0.664277,0.026227,0.792315,0.011444
1,word_unigram,0.669960,0.019630,0.789599,0.011126


Mean paired macro F1 delta: +0.005683


### Результат и интерпретация

На трёх folds unigram macro F1 составил `0.653685 / 0.697575 / 0.658621`; mean `0.669960`, std `0.019630`. Baseline mean был `0.664277`, поэтому mean paired delta равен `+0.005683`. На fold 0 unigram выше на `0.019131`, а на folds 1 и 2 ниже на `0.000779` и `0.001302`. Средняя accuracy unigram — `0.789599` против baseline `0.792315`. Фактический vocabulary содержит 4 959 / 5 000 / 5 000 признаков.

Вывод: unigram немного улучшил средний macro F1 на этих folds, но результат неоднороден между folds, а средняя accuracy снизилась. Это ограниченный сигнал для дальнейшего сравнения, не гарантия улучшения на outer validation или test. Общие folds и train-only fitting сохраняют честное сравнение.

### Самопроверка

1. Почему изменение только `ngram_range` помогает интерпретировать delta?
2. Как ограничение `max_features` может изменить эффект добавления bigrams?
3. Почему результаты трёх folds не гарантируют улучшение на outer validation или test?

## D4-05 — LinearSVC с прежним TF-IDF

**Гипотеза.** TF-IDF признаки и разбиения фиксированы, поэтому сравнение измеряет замену LogisticRegression на линейный margin classifier `LinearSVC`. `LinearSVC` оптимизирует squared hinge loss и возвращает классы через `predict`; вероятности `predict_proba` здесь не нужны. Все параметры из SPEC задаются явно, включая `dual="auto"` и `max_iter=5000`.

Обе модели запускаются через один runner на одинаковых folds. `ConvergenceWarning` делает запуск неуспешным и не должен скрываться или исправляться настройками, подобранными по test.

In [6]:
from finnews_sentiment.models.experiment_models import build_linear_svc_pipeline

linear_svc_pipeline = build_linear_svc_pipeline(config)
linear_svc_cv = run_cv_candidate(
    candidate_id="linear_svc",
    pipeline=linear_svc_pipeline,
    train=train_rows,
    fold_set=cv_folds,
)
linear_svc_paired = pd.DataFrame({
    "baseline_macro_f1": baseline_cv.fold_scores["macro_f1"],
    "linear_svc_macro_f1": linear_svc_cv.fold_scores["macro_f1"],
    "delta_macro_f1": linear_svc_cv.fold_scores["macro_f1"] - baseline_cv.fold_scores["macro_f1"],
    "baseline_accuracy": baseline_cv.fold_scores["accuracy"],
    "linear_svc_accuracy": linear_svc_cv.fold_scores["accuracy"],
    "baseline_features": baseline_cv.fold_scores["n_features"],
    "linear_svc_features": linear_svc_cv.fold_scores["n_features"],
    "convergence_status": linear_svc_cv.fold_scores["convergence_status"],
})
display(linear_svc_paired)
display(pd.DataFrame([
    {"candidate_id": baseline_cv.candidate_id, "mean_macro_f1": baseline_cv.mean_macro_f1,
     "std_macro_f1": baseline_cv.std_macro_f1, "mean_accuracy": baseline_cv.mean_accuracy,
     "std_accuracy": baseline_cv.std_accuracy},
    {"candidate_id": linear_svc_cv.candidate_id, "mean_macro_f1": linear_svc_cv.mean_macro_f1,
     "std_macro_f1": linear_svc_cv.std_macro_f1, "mean_accuracy": linear_svc_cv.mean_accuracy,
     "std_accuracy": linear_svc_cv.std_accuracy},
]))
print(f"Mean paired macro F1 delta: {linear_svc_paired['delta_macro_f1'].mean():+.6f}")

,baseline_macro_f1,linear_svc_macro_f1,delta_macro_f1,baseline_accuracy,linear_svc_accuracy,baseline_features,linear_svc_features,convergence_status
fold_id,,,,,,,,
0,0.634554,0.762359,0.127805,0.785617,0.835821,5000,5000,converged_no_warning
1,0.698353,0.800847,0.102494,0.808424,0.854620,5000,5000,converged_no_warning
2,0.659923,0.742939,0.083016,0.782904,0.819539,5000,5000,converged_no_warning


,candidate_id,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy
0,baseline,0.664277,0.026227,0.792315,0.011444
1,linear_svc,0.768715,0.024064,0.836660,0.014334


Mean paired macro F1 delta: +0.104438


### Результат и интерпретация

На folds 0/1/2 LinearSVC macro F1 составил `0.762359 / 0.800847 / 0.742939`; paired delta к baseline — `+0.127805 / +0.102494 / +0.083016`. Mean/std macro F1: `0.768715 / 0.024064` против baseline `0.664277 / 0.026227`. Mean/std accuracy: `0.836660 / 0.014334` против `0.792315 / 0.011444`. Обе модели использовали 5 000 TF-IDF features на каждом fold; все LinearSVC folds завершились `converged_no_warning`.

На этих folds замена классификатора дала более высокий macro F1 и accuracy во всех трёх частях. Это результат CV train folds и сам по себе не завершает выбор: validation confirmation предусмотрен позже, test остаётся закрытым.

### Самопроверка

1. Что осталось неизменным при сравнении с LogisticRegression и что заменили?
2. Почему `LinearSVC` score не является вероятностью?
3. Как трактовать результат, если folds расходятся или возникает ConvergenceWarning?

## D4-06 — ограниченный поиск LogisticRegression C

**Цель.** Проверить только `C ∈ {0.1, 1.0, 10.0}` при неизменных baseline TF-IDF и остальных параметрах LogisticRegression. `C` обратно связан с силой regularization: малый `C` сильнее ограничивает коэффициенты, большой `C` ослабляет ограничение.

Полный Pipeline передаётся в `GridSearchCV`; поэтому в каждом fold TF-IDF fit-ится заново только на train portion. Переданы сохранённые folds, `refit=False` и `error_score="raise"`. Primary scoring — macro F1 с фиксированными labels и `zero_division=0`; accuracy сохраняется дополнительно. Поиск не получает outer validation/test. Grid фиксирован заранее: выбор лучших CV score после нескольких вариантов всё равно может подогнаться к этим folds.

In [7]:
import numpy as np

from finnews_sentiment.models.cv_search import search_logreg_c

c_search = search_logreg_c(
    pipeline=build_baseline_pipeline(config),
    train=train_rows,
    fold_set=cv_folds,
)
c_search_scores = c_search.scores
c_one_search = c_search_scores.loc[c_search_scores["C"].eq(1.0)].iloc[0]
np.testing.assert_allclose(
    [c_one_search[f"fold_{fold_id}_macro_f1"] for fold_id in range(cv_folds.n_splits)],
    baseline_cv.fold_scores["macro_f1"].to_numpy(),
)
np.testing.assert_allclose(
    [c_one_search[f"fold_{fold_id}_accuracy"] for fold_id in range(cv_folds.n_splits)],
    baseline_cv.fold_scores["accuracy"].to_numpy(),
)
best_c_row = c_search_scores.loc[c_search_scores["mean_macro_f1"].idxmax()]
display(c_search_scores.round(6))
print(f"Best mean CV macro F1 in this grid: C={best_c_row['C']:.1f}")
print(f"Warnings: {c_search.warnings}")

,C,fold_0_macro_f1,fold_1_macro_f1,fold_2_macro_f1,fold_0_accuracy,fold_1_accuracy,fold_2_accuracy,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy,mean_fit_seconds
0,0.1,0.413144,0.375433,0.355490,0.679783,0.676630,0.662144,0.381356,0.023907,0.672852,0.007681,1.367992
1,1.0,0.634554,0.698353,0.659923,0.785617,0.808424,0.782904,0.664277,0.026227,0.792315,0.011444,3.375210
2,10.0,0.744646,0.791854,0.741290,0.826323,0.851902,0.819539,0.759263,0.023086,0.832588,0.013935,4.575577


Best mean CV macro F1 in this grid: C=10.0
Warnings: ()


### Результат и интерпретация

Grid дал macro F1 folds и mean/std: для `C=0.1` — `0.413144 / 0.375433 / 0.355490`, `0.381356 / 0.023907`; для `C=1.0` — `0.634554 / 0.698353 / 0.659923`, `0.664277 / 0.026227`; для `C=10.0` — `0.744646 / 0.791854 / 0.741290`, `0.759263 / 0.023086`. Значения `C=1.0` совпали с D4-03 baseline. `C=10.0` имеет наибольший mean macro F1 в этом grid. Mean accuracy по тем же конфигурациям: `0.672852 / 0.792315 / 0.832588`. Warnings отсутствовали.

Пока можно заключить лишь, что среди трёх проверенных значений `C=10.0` имеет наивысший средний CV macro F1. Это не финальная оценка и не выбор среди всех Day04 кандидатов: сравнение и tie-breaking выполняются в D4-07. Перебор нескольких `C` повышает риск адаптации к folds, поэтому лучший CV score не заменяет одно validation confirmation. Test не участвовал в подборе.

### Самопроверка

1. Что происходит с regularization при уменьшении `C`?
2. Почему `C=1.0` должен повторить D4-03 baseline?
3. Почему максимум по CV после перебора не равен финальной test оценке?